# 2048 AI - Snake N-Tuple Trainer

Notebook này đã được dọn sạch. Model đang dùng để chơi local vẫn là `js/ai/ntuple/ntupleWeights.js` với bot `ntupleTdBeam2`, `futureWeight=0.49`. Phần Colab chỉ còn dùng để train model mới `snake6`.

## Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import shutil

PROJECT = Path('/content/drive/MyDrive/MyProjects/2048-ai')

targets = [
    PROJECT / 'models/ntupleWeights-paper17.js',
    PROJECT / 'models/ntupleWeights-paper17-early2048.js',
    PROJECT / 'models/ntupleWeights-paper17-auto-mix.js',
    PROJECT / 'models/ntupleWeights-medium.js',

    PROJECT / 'logs/ntuple_small_100k.log',
    PROJECT / 'logs/ntuple_small_resume.log',
    PROJECT / 'logs/ntuple_medium_100k.log',
    PROJECT / 'logs/ntuple_medium_resume.log',
    PROJECT / 'logs/ntuple_paper17_100k.log',
    PROJECT / 'logs/ntuple_paper17_resume.log',
    PROJECT / 'logs/ntuple_paper17_early2048.log',
    PROJECT / 'logs/ntuple_paper17_early2048_resume.log',
    PROJECT / 'logs/ntuple_paper17_auto_mix.log',
    PROJECT / 'logs/ntuple_paper17_auto_mix_resume.log',

    PROJECT / 'data/move-ranking-v1.jsonl',
    PROJECT / 'data/move-policy-v1.jsonl',
    PROJECT / 'data/afterstate-value-v1.jsonl',
    PROJECT / 'data/states-at-2048.jsonl',
    PROJECT / 'data/states-at-4096.jsonl',
    PROJECT / 'data/states-at-8192.jsonl',
    PROJECT / 'data/states-at-16384.jsonl',
    PROJECT / 'data/states-at-32768.jsonl',
    PROJECT / 'data/failed-before-2048.jsonl',
    PROJECT / 'data/failed-before-4096.jsonl',
    PROJECT / 'data/failed-before-8192.jsonl',
    PROJECT / 'data/failed-before-16384.jsonl',
    PROJECT / 'data/failed-before-32768.jsonl',

    PROJECT / 'outputs/checkpoints-paper17',
    PROJECT / 'outputs/checkpoints-paper17-early2048',
    PROJECT / 'outputs/checkpoints-paper17-auto-mix',
    PROJECT / 'outputs/checkpoints-medium',
    PROJECT / 'outputs/checkpoints-small',
]

deleted = []
missing = []

for target in targets:
    if target.exists():
        if target.is_dir():
            shutil.rmtree(target)
        else:
            target.unlink()
        deleted.append(str(target))
    else:
        missing.append(str(target))

print('Deleted:', len(deleted))
for item in deleted:
    print(' -', item)

print('\nMissing/skipped:', len(missing))
for item in missing:
    print(' -', item)

print('\nKept important files if they exist:')
for item in [
    PROJECT / 'models/ntupleWeights.js',
    PROJECT / 'models/ntupleWeights-snake6-td.js',
    PROJECT / 'scripts/train_ntuple_td.py',
    PROJECT / 'scripts/trainSnakeColab.sh',
]:
    print(item, 'exists=', item.exists(), 'size=', item.stat().st_size if item.exists() else 0)

Deleted: 23
 - /content/drive/MyDrive/MyProjects/2048-ai/models/ntupleWeights-paper17.js
 - /content/drive/MyDrive/MyProjects/2048-ai/models/ntupleWeights-paper17-early2048.js
 - /content/drive/MyDrive/MyProjects/2048-ai/models/ntupleWeights-paper17-auto-mix.js
 - /content/drive/MyDrive/MyProjects/2048-ai/models/ntupleWeights-medium.js
 - /content/drive/MyDrive/MyProjects/2048-ai/logs/ntuple_small_100k.log
 - /content/drive/MyDrive/MyProjects/2048-ai/logs/ntuple_small_resume.log
 - /content/drive/MyDrive/MyProjects/2048-ai/logs/ntuple_medium_100k.log
 - /content/drive/MyDrive/MyProjects/2048-ai/logs/ntuple_medium_resume.log
 - /content/drive/MyDrive/MyProjects/2048-ai/logs/ntuple_paper17_100k.log
 - /content/drive/MyDrive/MyProjects/2048-ai/logs/ntuple_paper17_resume.log
 - /content/drive/MyDrive/MyProjects/2048-ai/logs/ntuple_paper17_early2048.log
 - /content/drive/MyDrive/MyProjects/2048-ai/logs/ntuple_paper17_auto_mix_resume.log
 - /content/drive/MyDrive/MyProjects/2048-ai/data/stat

## Setup project paths

In [ ]:
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/MyProjects/2048-ai')
DATA = PROJECT / 'data'
LOGS = PROJECT / 'logs'
MODELS = PROJECT / 'models'
OUTPUTS = PROJECT / 'outputs'
SCRIPTS = PROJECT / 'scripts'

for folder in (DATA, LOGS, MODELS, OUTPUTS, SCRIPTS):
    folder.mkdir(parents=True, exist_ok=True)

print('Project:', PROJECT)
print('Trainer:', SCRIPTS / 'train_ntuple_td.py')
print('Snake script:', SCRIPTS / 'trainSnakeColab.sh')

## Upload trainer files

Upload 2 file này từ project local:

- `js/ai/ntuple/train_ntuple_td.py`
- `js/ai/colab/trainSnakeColab.sh`

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()

required = {'train_ntuple_td.py', 'trainSnakeColab.sh'}
missing = required - set(uploaded)
if missing:
    raise RuntimeError(f'Missing upload files: {sorted(missing)}')

shutil.copy('/content/train_ntuple_td.py', SCRIPTS / 'train_ntuple_td.py')
shutil.copy('/content/trainSnakeColab.sh', SCRIPTS / 'trainSnakeColab.sh')

print('Saved trainer:', SCRIPTS / 'train_ntuple_td.py')
print('Saved script:', SCRIPTS / 'trainSnakeColab.sh')

## Train snake6

Cell này tự resume nếu `models/ntupleWeights-snake6-td.js` đã tồn tại.

In [ ]:
!chmod +x "$SCRIPTS/trainSnakeColab.sh"
!EPISODES=200000 "$SCRIPTS/trainSnakeColab.sh"

## Copy model to outputs

Dùng cell này để tạo bản copy có timestamp rồi tải tay từ Google Drive nếu cần.

In [ ]:
import shutil, time

src = MODELS / 'ntupleWeights-snake6-td.js'
stamp = time.strftime('%Y%m%d-%H%M%S')
dst = OUTPUTS / f'ntupleWeights-snake6-td-{stamp}.js'

if not src.exists():
    raise FileNotFoundError(src)

shutil.copy(src, dst)
print('Copied:', dst)
print('Size:', dst.stat().st_size)